In [1]:
import pandas as pd

df = pd.read_csv('sales_data.csv')
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


### Exploring Data

---

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                76000 non-null  str    
 1   Store ID            76000 non-null  str    
 2   Product ID          76000 non-null  str    
 3   Category            76000 non-null  str    
 4   Region              76000 non-null  str    
 5   Inventory Level     76000 non-null  int64  
 6   Units Sold          76000 non-null  int64  
 7   Units Ordered       76000 non-null  int64  
 8   Price               76000 non-null  float64
 9   Discount            76000 non-null  int64  
 10  Weather Condition   76000 non-null  str    
 11  Promotion           76000 non-null  int64  
 12  Competitor Pricing  76000 non-null  float64
 13  Seasonality         76000 non-null  str    
 14  Epidemic            76000 non-null  int64  
 15  Demand              76000 non-null  int64  
dtypes: float64(2), 

In [2]:
df.shape

(76000, 16)

In [3]:
df.isna().sum()

Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Price                 0
Discount              0
Weather Condition     0
Promotion             0
Competitor Pricing    0
Seasonality           0
Epidemic              0
Demand                0
dtype: int64

In [7]:
# detect categorical columns and compute unique values
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
    if col == "Date":
        continue
    print(col)
    print(df[col].unique().tolist())
    print("=" * 40)

Store ID
['S001', 'S002', 'S003', 'S004', 'S005']
Product ID
['P0001', 'P0002', 'P0003', 'P0004', 'P0005', 'P0006', 'P0007', 'P0008', 'P0009', 'P0010', 'P0011', 'P0012', 'P0013', 'P0014', 'P0015', 'P0016', 'P0017', 'P0018', 'P0019', 'P0020']
Category
['Electronics', 'Clothing', 'Groceries', 'Toys', 'Furniture']
Region
['North', 'South', 'East', 'West']
Weather Condition
['Snowy', 'Cloudy', 'Sunny', 'Rainy']
Seasonality
['Winter', 'Spring', 'Summer', 'Autumn']


C:\Users\madda\AppData\Local\Temp\ipykernel_12604\2949373271.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()


### Feature Engineering

---


In [9]:
import numpy as np

In [10]:
def engineer_features(file_path):
    # 1. Load the dataset
    df = pd.read_csv(file_path)
    
    # Ensure Date column is explicitly parsed as datetime
    df['Date'] = pd.to_datetime(df['Date'])
    
    # CRITICAL: Sort chronologically by Store, Product, and Date for accurate lag calculation
    df = df.sort_values(by=['Store ID', 'Product ID', 'Date']).reset_index(drop=True)
    
    print("--- Generating Features ---")
    
    # ==========================================
    # 1. TEMPORAL FEATURES
    # ==========================================
    df['Day_of_Week'] = df['Date'].dt.dayofweek          # Monday=0, Sunday=6
    df['Month'] = df['Date'].dt.month                  # 1 to 12
    df['Quarter'] = df['Date'].dt.quarter              # 1 to 4
    df['Is_Weekend'] = df['Day_of_Week'].isin([5, 6]).astype(int)
    df['Is_Month_Start'] = df['Date'].dt.is_month_start.astype(int)
    df['Is_Month_End'] = df['Date'].dt.is_month_end.astype(int)
    
    # ==========================================
    # 2. LAG FEATURES (Targeting 'Demand')
    # ==========================================
    # Using groupby ensure metrics are isolated per store-product stream
    grouped_series = df.groupby(['Store ID', 'Product ID'])['Demand']
    
    df['Demand_Lag_1'] = grouped_series.shift(1)
    df['Demand_Lag_7'] = grouped_series.shift(7)
    
    # Note: To avoid data leakage, we shift the series by 1 day BEFORE calculating the 7-day rolling mean.
    # This ensures today's rolling baseline only contains data up to yesterday.
    df['Rolling_Mean_7'] = grouped_series.transform(lambda x: x.shift(1).rolling(window=7).mean())
    
    # ==========================================
    # 3. BUSINESS-DRIVEN DERIVED FEATURES
    # ==========================================
    # Price difference: positive means we are more expensive, negative means we are cheaper
    df['Price_Diff'] = df['Price'] - df['Competitor Pricing']
    
    # Convert nominal discount values (e.g., 5, 15) into a decimal rate
    df['Discount_Rate'] = df['Discount'] / 100.0
    
    # Event tracking flag: Is there an active promotion running on a weekend?
    df['Promoted_Weekend'] = ((df['Promotion'] == 1) & (df['Is_Weekend'] == 1)).astype(int)
    
    # ==========================================
    # 4. POST-PROCESSING CLEANUP
    # ==========================================
    # Shifting windows natively generate NaN values for the initial records of each group.
    # Out of 76,000 rows, losing 700 rows (7 rows * 100 unique items) is mathematically safer than imputing zeroes.
    initial_shape = df.shape[0]
    df = df.dropna().reset_index(drop=True)
    final_shape = df.shape[0]
    
    print(f"Features created successfully!")
    print(f"Dropped {initial_shape - final_shape} rows containing lag-induced NaN values.")
    print(f"Final training-ready dataset shape: {df.shape}\n")
    
    return df

# Run the pipeline
engineered_df = engineer_features('sales_data.csv')

--- Generating Features ---
Features created successfully!
Dropped 700 rows containing lag-induced NaN values.
Final training-ready dataset shape: (75300, 28)



In [11]:
engineered_df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Quarter,Is_Weekend,Is_Month_Start,Is_Month_End,Demand_Lag_1,Demand_Lag_7,Rolling_Mean_7,Price_Diff,Discount_Rate,Promoted_Weekend
0,2022-01-08,S001,P0001,Electronics,North,308,96,248,68.56,10,...,1,1,0,0,87.0,115.0,105.857143,-12.43,0.10,1
1,2022-01-09,S001,P0001,Electronics,North,212,57,0,62.76,10,...,1,1,0,0,113.0,84.0,105.571429,-4.35,0.10,0
2,2022-01-10,S001,P0001,Electronics,North,403,88,0,68.99,5,...,1,0,0,0,87.0,132.0,106.000000,-12.15,0.05,0
3,2022-01-11,S001,P0001,Electronics,North,315,112,0,73.71,0,...,1,0,0,0,99.0,67.0,101.285714,-11.58,0.00,0
4,2022-01-12,S001,P0001,Electronics,North,203,70,179,65.76,10,...,1,0,0,0,103.0,110.0,106.428571,5.63,0.10,0


### Find the cols with negative values

---

In [13]:
# Find negative numeric values in df and engineered_df

# select numeric columns (no re-imports needed)
numeric_cols_df = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_eng = engineered_df.select_dtypes(include=[np.number]).columns.tolist()

def find_negatives(frame, name):
    neg_info = {}
    for col in frame.select_dtypes(include=[np.number]).columns:
        neg_vals = frame.loc[frame[col] < 0, col].unique()
        if neg_vals.size > 0:
            neg_info[col] = neg_vals
    print(f"\n{name} - columns with negative values: {len(neg_info)}")
    for k, v in neg_info.items():
        print(f"{k}: {v}")
    neg_rows = frame[frame.select_dtypes(include=[np.number]).lt(0).any(axis=1)]
    print(f"{name} - rows with any negative numeric value: {len(neg_rows)}")
    display(neg_rows.head(5))  # show up to first 50 offending rows

find_negatives(df, "df")
find_negatives(engineered_df, "engineered_df")


df - columns with negative values: 0
df - rows with any negative numeric value: 0


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand



engineered_df - columns with negative values: 1
Price_Diff: [-12.43  -4.35 -12.15 ...  -6.64  -1.79  -3.32]
engineered_df - rows with any negative numeric value: 37462


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Quarter,Is_Weekend,Is_Month_Start,Is_Month_End,Demand_Lag_1,Demand_Lag_7,Rolling_Mean_7,Price_Diff,Discount_Rate,Promoted_Weekend
0,2022-01-08,S001,P0001,Electronics,North,308,96,248,68.56,10,...,1,1,0,0,87.0,115.0,105.857143,-12.43,0.10,1
1,2022-01-09,S001,P0001,Electronics,North,212,57,0,62.76,10,...,1,1,0,0,113.0,84.0,105.571429,-4.35,0.10,0
2,2022-01-10,S001,P0001,Electronics,North,403,88,0,68.99,5,...,1,0,0,0,87.0,132.0,106.000000,-12.15,0.05,0
3,2022-01-11,S001,P0001,Electronics,North,315,112,0,73.71,0,...,1,0,0,0,99.0,67.0,101.285714,-11.58,0.00,0
5,2022-01-13,S001,P0001,Electronics,North,133,80,0,54.01,20,...,1,0,0,0,85.0,146.0,102.857143,-7.34,0.20,0


#### <u>Observation</u>: 
**In the feature engineering step, we defined the calculation as:**
`Price_Diff = Price - Competitor Pricing` 

**When this value is negative, it mathematically means your price is lower than the competitor's price (`Price < Competitor Pricing`).**

---

### <u>Creating more features for Sales</u>:



In [16]:
# 1. Financial Revenue Calculation
engineered_df['Revenue'] = (
    engineered_df['Units Sold'] * engineered_df['Price'] * (1 - engineered_df['Discount_Rate'])
)

# 2. Inventory Buffer Ratio
# Adding a minor constant (1e-5) avoids potential division-by-zero errors if rolling mean is 0
engineered_df['Inventory_Buffer_Ratio'] = (
    engineered_df['Inventory Level'] / 
    (engineered_df['Rolling_Mean_7'] + 1e-5)
)

# 3. Normalized Price Ratio
engineered_df['Price_Ratio'] = engineered_df['Price'] / engineered_df['Competitor Pricing']


print("Final feature set compiled. Ready for Model Training & Dashboard Engineering!")
engineered_df.head()

Final feature set compiled. Ready for Model Training & Dashboard Engineering!


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Is_Month_End,Demand_Lag_1,Demand_Lag_7,Rolling_Mean_7,Price_Diff,Discount_Rate,Promoted_Weekend,Revenue,Inventory_Buffer_Ratio,Price_Ratio
0,2022-01-08,S001,P0001,Electronics,North,308,96,248,68.56,10,...,0,87.0,115.0,105.857143,-12.43,0.10,1,5923.584,2.909581,0.846524
1,2022-01-09,S001,P0001,Electronics,North,212,57,0,62.76,10,...,0,113.0,84.0,105.571429,-4.35,0.10,0,3219.588,2.008119,0.935181
2,2022-01-10,S001,P0001,Electronics,North,403,88,0,68.99,5,...,0,87.0,132.0,106.000000,-12.15,0.05,0,5767.564,3.801886,0.850259
3,2022-01-11,S001,P0001,Electronics,North,315,112,0,73.71,0,...,0,99.0,67.0,101.285714,-11.58,0.00,0,8255.520,3.110014,0.864228
4,2022-01-12,S001,P0001,Electronics,North,203,70,179,65.76,10,...,0,103.0,110.0,106.428571,5.63,0.10,0,4142.880,1.907382,1.093630


In [17]:
engineered_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 75300 entries, 0 to 75299
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    75300 non-null  datetime64[us]
 1   Store ID                75300 non-null  str           
 2   Product ID              75300 non-null  str           
 3   Category                75300 non-null  str           
 4   Region                  75300 non-null  str           
 5   Inventory Level         75300 non-null  int64         
 6   Units Sold              75300 non-null  int64         
 7   Units Ordered           75300 non-null  int64         
 8   Price                   75300 non-null  float64       
 9   Discount                75300 non-null  int64         
 10  Weather Condition       75300 non-null  str           
 11  Promotion               75300 non-null  int64         
 12  Competitor Pricing      75300 non-null  float64       
 1

In [18]:
engineered_df.to_csv("final_data.csv")